# Modern CNN Architectures (Residual Networks) — AI Lab Instructor Notebook

*Deep Learning · Hard*



4 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# Modern CNN Architectures — Residual Networks (ResNet) — Student Lab

We focus on the core idea: **residual connections improve gradient flow**, enabling very deep networks.

## Section 0 — Digits baseline

Start with a baseline on flattened pixels.

In [ ]:
import numpy as np

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

digits = load_digits()
X_img = digits.images.astype(float)
y = digits.target.astype(int)

X_flat = X_img.reshape(len(X_img), -1)
X_tr, X_va, y_tr, y_va = train_test_split(X_flat, y, test_size=0.25, random_state=0, stratify=y)
baseline = Pipeline(steps=[('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=5000))])
baseline.fit(X_tr, y_tr)
print('baseline train acc', float(baseline.score(X_tr, y_tr)))
print('baseline val   acc', float(baseline.score(X_va, y_va)))
check('img_shape', X_img.shape[1:] == (8, 8))

**Why this works.** # Modern CNN Architectures — Residual Networks (ResNet) — Instructor Lab

Solutions + short rationales. Focus: gradient flow intuition and a minimal residual block forward pass.

## Section 0 — Digits baseline

Rationale: always compare to a simple, strong baseline.

## Task 2: Implement `simulate_grad_norms(depth, d, mode)` returning an array of gradient norms at each layer.
*Section: Gradient flow simulation (why ResNets help)*

## Section 1 — Gradient flow simulation (why ResNets help)

We simulate backprop *gradient norm* through a deep network with random linear layers.

Key idea:
- **Plain**: `h_{l+1} = W_l h_l`
- **Residual**: `h_{l+1} = h_l + W_l h_l`

In backward, the Jacobians multiply. Residual blocks add an identity term that prevents norms from collapsing too fast.

Important: the returned array is ordered from **input side -> loss side**.
So if early-layer gradients vanish, the first value should be much smaller than the last value.

### Task 1.1
Implement `simulate_grad_norms(depth, d, mode)` returning an array of gradient norms at each layer.

In [ ]:
$$f

In [ ]:
# Checks
check('finite_plain', np.isfinite(plain).all())
check('finite_resid', np.isfinite(resid).all())
check('plain_vanishes_toward_input', plain[0] < 0.1 * plain[-1])
check('resid_stays_healthier', resid[0] > 0.5 * resid[-1] and resid[0] < 10.0 * resid[-1])

**Why this works.** ## Section 1 — Gradient flow simulation

Rationale: In backprop, layer Jacobians multiply. Residual blocks introduce an identity term, keeping gradients healthier.

## Task 3: Implement `conv2d_out_shape` and `conv2d_forward` (NCHW).
*Section: CNN building blocks (forward-only)*

## Section 2 — CNN building blocks (forward-only)

We implement a small forward-only residual block using convs:
- `y = x + conv(relu(conv(x)))`

We require shapes to match for the skip connection (same channels, same H/W).

### Task 2.1
Implement `conv2d_out_shape` and `conv2d_forward` (NCHW).

### Task 2.2
Implement `residual_block_forward`.

In [ ]:
$$10

In [ ]:
# Checks
check('res_block_shape', y_block.shape == x.shape)

**Why this works.** ## Section 2 — Residual block forward (conv)

Rationale: a residual block learns a correction to identity: `y = x + F(x)`.

## Task 4: Implement `plain_stack_features` and `residual_stack_features`.
*Section: Plain vs ResNet-ish random features*

## Section 3 — Plain vs ResNet-ish random features

We compare two *random* feature extractors (forward-only):
- **Plain deep conv stack**: repeated `conv → ReLU`
- **Residual deep conv stack**: repeated residual blocks

We then train LogisticRegression on the extracted features.

### Task 3.1
Implement `plain_stack_features` and `residual_stack_features`.

HINT: keep channels small (e.g., 8 or 16) to keep this fast.

**Hints**
- keep channels small (e.g., 8 or 16) to keep this fast.

In [ ]:
$$11

In [ ]:
# Checks
check('val_acc_sane_plain', 0.6 <= va_p <= 1.0)
check('val_acc_sane_resid', 0.6 <= va_r <= 1.0)

**Why this works.** ## Section 3 — Plain vs residual random features

Rationale: forward-only random conv stacks can already separate digits somewhat; residual stacks tend to be numerically more stable at depth.